# 02 · Cleaning Is a Sequence of Decisions

Every cleaning step throws information away or invents it. Neither is free, so
each one is a decision you should be able to defend.

This notebook works through the problems found in notebook 01. For each, the
question is not "what's the pandas command" but **"what am I claiming about the
world by doing this?"**

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

DATA = Path.cwd()
while not (DATA / "data" / "turbine_readings.csv").exists() and DATA != DATA.parent:
    DATA = DATA.parent
CSV = DATA / "data" / "turbine_readings.csv"
print("reading", CSV.relative_to(DATA))

reading data/turbine_readings.csv


In [2]:
df = pd.read_csv(CSV, parse_dates=["install_date"])
print(df.shape)

(6150, 11)


## Decision 1 · Duplicates

Exact duplicates are almost always a logging artefact. Dropping them is safe.

Rows identical apart from the id are a judgement call: either the same turbine was
logged twice, or two turbines genuinely share every reading. Given continuous sensor
values, coincidence is implausible.

In [3]:
before = len(df)
df = df.drop_duplicates()
print(f"exact duplicates removed : {before - len(df)}")

cols = [c for c in df.columns if c != "turbine_id"]
before = len(df)
df = df.drop_duplicates(subset=cols)
print(f"re-logged rows removed   : {before - len(df)}")
print(f"remaining                : {len(df)}")

exact duplicates removed : 66
re-logged rows removed   : 49
remaining                : 6035


## Decision 2 · The units problem

`gearbox_temp` runs from ~30 to ~234 in a column labelled as one unit. Two sites
are reporting Fahrenheit.

In [4]:
df.groupby("site")["gearbox_temp"].agg(["min", "median", "max", "count"]).round(1)

,min,median,max,count
site,,,,
Ardrossan,29.8,71.0,108.2,2155
Beinn an Tuirc,99.1,160.3,232.3,897
Clyde,88.5,161.5,234.3,1218
Whitelee,32.5,71.0,113.5,1765


Clyde and Beinn an Tuirc sit in a completely different range. Nothing in the
column name says so — the only way to catch this is to look at the distribution
per group.

Convert those two sites to Celsius. Note what we're assuming: that the *site* is what
determines the unit, and that no site switched partway through. Both are worth
confirming with whoever owns the data — and in a real project, that conversation
happens before this cell is written.

In [5]:
F_SITES = ["Clyde", "Beinn an Tuirc"]
mask = df["site"].isin(F_SITES)
df.loc[mask, "gearbox_temp"] = ((df.loc[mask, "gearbox_temp"] - 32) * 5 / 9).round(1)
df.rename(columns={"gearbox_temp": "gearbox_temp_c"}, inplace=True)

df.groupby("site")["gearbox_temp_c"].agg(["min", "median", "max"]).round(1)

,min,median,max
site,,,
Ardrossan,29.8,71.0,108.2
Beinn an Tuirc,37.3,71.3,111.3
Clyde,31.4,72.0,112.4
Whitelee,32.5,71.0,113.5


The ranges now agree. Renaming the column to carry its unit is cheap insurance
against exactly this bug recurring.

## Decision 3 · Parsing the string column

In [6]:
df["operating_hours"] = (
    df["operating_hours"].astype(str)
      .str.replace(",", "", regex=False)
      .str.replace(r"\s*hrs?\s*$", "", regex=True)
      .replace({"nan": np.nan, "None": np.nan})
      .astype(float)
)
df["operating_hours"].describe().round(0)

count      5659.0
mean      34824.0
std       19731.0
min         841.0
25%       20375.0
50%       31230.0
75%       45466.0
max      146920.0
Name: operating_hours, dtype: float64

## Decision 4 · The sensor glitches

`rotor_speed_rpm` contains values of exactly 9999.0. A repeated, round, physically
impossible number is a **sentinel** — a placeholder some system writes when a reading
fails. It is missing data wearing a costume.

Treating it as a real value would badly distort every statistic. Treating it as
missing is the honest move.

In [7]:
glitch = df["rotor_speed_rpm"] == 9999.0
print(f"sentinel values: {glitch.sum()}")
print(f"mean rotor speed WITH them   : {df['rotor_speed_rpm'].mean():.1f}")
df.loc[glitch, "rotor_speed_rpm"] = np.nan
print(f"mean rotor speed WITHOUT them: {df['rotor_speed_rpm'].mean():.1f}")

sentinel values: 22
mean rotor speed WITH them   : 53.3
mean rotor speed WITHOUT them: 14.4


## Decision 5 · Standardising the inspector names

In [8]:
def norm_name(v):
    if pd.isna(v):
        return v
    v = v.lower().replace(".", " ").replace("_", " ")
    parts = [p for p in v.split() if p]
    return f"{parts[0][0]}.{parts[-1]}" if len(parts) > 1 else parts[0]

print(f"before: {df['last_inspector'].nunique()} distinct")
df["last_inspector"] = df["last_inspector"].map(norm_name)
print(f"after : {df['last_inspector'].nunique()} distinct")
df["last_inspector"].value_counts(dropna=False)

before: 6 distinct
after : 3 distinct


last_inspector
j.okafor       1877
s.tran         1875
m.lindqvist    1795
NaN             488
Name: count, dtype: int64

## Decision 6 · Missing values, and why the mechanism matters

The most common mistake here is filling gaps with the column mean without asking
**why** the values are missing. Three mechanisms, three consequences:

| Mechanism | Meaning | Consequence |
|---|---|---|
| MCAR | Missing completely at random | Filling is relatively safe |
| MAR | Missingness explained by *other* columns | Fill using those columns |
| MNAR | Missingness relates to the missing value itself | Filling introduces real bias |

Test whether vibration is missing at random by comparing rows where it's absent
against rows where it's present.

In [9]:
flag = df["vibration_mm_s"].isna()
comp = df.groupby(flag)[["age_years", "gearbox_temp_c", "failed_within_90d"]].mean().round(3)
comp.index = ["vibration present", "vibration MISSING"]
comp

,age_years,gearbox_temp_c,failed_within_90d
vibration present,4.754,71.134,0.149
vibration MISSING,4.884,75.021,0.240


Rows with missing vibration have a **substantially higher failure rate**. The
values are not missing at random — the sensor tends to drop out on the turbines that
are shaking hardest.

That has two implications:

1. Filling with the median would pull the worst turbines toward normal and destroy
   the signal you care about most.
2. **The missingness is itself predictive.** Record it as a feature before filling.

In [10]:
df["vibration_missing"] = df["vibration_mm_s"].isna().astype(int)

print("failure rate by missingness flag:")
print(df.groupby("vibration_missing")["failed_within_90d"].mean().round(3))

failure rate by missingness flag:
vibration_missing
0    0.149
1    0.240
Name: failed_within_90d, dtype: float64


> **Do not impute here.** Any filling of `vibration_mm_s` must be fitted on the
> training split only, inside a pipeline — otherwise the test set's statistics leak
> into training. That's notebook 04's job, and doing it now would be the classic
> preprocessing leak.

## Decision 7 · Drop the leaking column

In [11]:
df = df.drop(columns=["maintenance_ticket_raised"])
print("dropped maintenance_ticket_raised — it encodes the outcome")

dropped maintenance_ticket_raised — it encodes the outcome


## Save the cleaned data

In [12]:
out = CSV.parent / "turbine_clean.csv"
df.to_csv(out, index=False)
print(f"wrote {out.name}  {df.shape[0]} rows x {df.shape[1]} cols")
df.dtypes

wrote turbine_clean.csv  6035 rows x 11 cols


turbine_id                      str
site                            str
install_date         datetime64[us]
age_years                   float64
rotor_speed_rpm             float64
vibration_mm_s              float64
gearbox_temp_c              float64
operating_hours             float64
last_inspector                  str
failed_within_90d             int64
vibration_missing             int64
dtype: object

---

## What you decided

Every one of these is defensible and none is automatic:

- dropped duplicates, including re-logged rows
- converted two sites from Fahrenheit, **assuming site determines unit**
- treated 9999.0 as missing rather than real
- collapsed inspector name variants
- kept missingness as a feature instead of filling it
- dropped a leaking column

Write the assumptions down. When the model behaves oddly in three weeks, this list
is where you look first.

## Your turn

1. `install_date` and `age_years` should agree. Do they? What would you do if not?
2. Is `operating_hours` consistent with `age_years`? Any turbine running more hours
   than it has existed?
3. What would change if a site had switched from Fahrenheit to Celsius mid-year?
   How would you detect that?